In [ ]:
import os
import re
import ast
import json
import difflib
import astunparse
import anthropic
from dotenv import load_dotenv
from openai import OpenAIError
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers.json import JsonOutputParser
from langchain.schema.output_parser import StrOutputParser
from langchain.schema import HumanMessage
from datasets import load_dataset
from langchain.llms.base import BaseLLM
from langchain.chat_models.base import BaseChatModel
import multiprocessing
from functools import partial
import time
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

In [ ]:
load_dotenv()

dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

# Define LLMs with API keys loaded from environment variables
llms = {
    "chatgpt": ChatOpenAI(
        model_name="gpt-4o",
        openai_api_key=os.getenv("OPENAI_API_KEY")
    ),
    "claude": ChatAnthropic(
        model_name="claude-3-5-sonnet-20240620",
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY")
    ),
    "deepseek": ChatDeepSeek(
        model="deepseek-chat",
        api_key=os.getenv("DEEPSEEK_API_KEY")
    )
}

In [ ]:
json_parser = JsonOutputParser()

# AST-based prompt template
ast_prompt_template = PromptTemplate(
    input_variables=["problem_statement", "ast_content", "file_path"],
    template="""
We are solving the following issue:
--- BEGIN ISSUE ---
{problem_statement}
--- END ISSUE ---

Below is the Abstract Syntax Tree (AST) representation of the file that needs to be fixed:
--- BEGIN AST ---
{ast_content}
--- END AST ---

The original file path is: {file_path}

Instructions:
1. Analyze the problem statement and the AST carefully.
2. Identify the parts of the AST that need to be modified to fix the issue.
3. For each part that needs fixing, extract the complete subtree that contains the bug.
4. Create fixed versions of each identified subtree.
5. Return a JSON list where each element is a string representation of a fixed AST subtree.

Important guidelines:
- Each AST subtree should be complete and syntactically valid.
- Include enough context in each subtree to clearly identify where it belongs in the original AST.
- Don't make the subtrees too large (no more than needed to fix the issue).
- Don't make them too small (include enough context to locate them).
- Ensure the JSON output follows this exact format: ["<fixed AST subtree 1>", "<fixed AST subtree 2>", ...]

The output MUST be a valid JSON list of strings that can be parsed directly.
"""
)

In [ ]:
def ast_to_string(node):
    """Convert an AST node to a string representation."""
    return ast.dump(node)

def string_to_ast(ast_string):
    """Convert a string representation back to an AST node."""
    return ast.parse(ast_string, mode='eval').body

def extract_modified_file_path(patch):
    """Extracts the modified file path from the first line of a Git diff."""
    match = re.search(r'diff --git a/(.*?) b/', patch)
    return match.group(1) if match else None

In [ ]:
def find_matching_node(original_ast, target_ast_str):
    """
    Find a node in the original AST that matches the pattern in the target AST string.
    Returns the node and its parent if found, otherwise None.
    """
    target = ast.parse(target_ast_str.strip())
    target_dump = ast.dump(target)
    
    class NodeFinder(ast.NodeVisitor):
        def __init__(self):
            self.results = []
            self.current_parent = None
            
        def generic_visit(self, node):
            previous_parent = self.current_parent
            for field, value in ast.iter_fields(node):
                if isinstance(value, list):
                    for idx, item in enumerate(value):
                        if isinstance(item, ast.AST):
                            self.current_parent = (node, field, idx)
                            self.visit(item)
                elif isinstance(value, ast.AST):
                    self.current_parent = (node, field, None)
                    self.visit(value)
            
            # Check if this node matches our target
            node_dump = ast.dump(node)
            if node_dump == target_dump:
                self.results.append((node, self.current_parent))
                
            self.current_parent = previous_parent
    
    finder = NodeFinder()
    finder.visit(original_ast)
    return finder.results

In [ ]:
def replace_node_in_ast(original_ast, parent_info, new_node):
    """
    Replace a node in the original AST with a new node.
    parent_info is a tuple of (parent_node, field_name, index)
    """
    parent, field, idx = parent_info
    
    if idx is not None:  # Node is part of a list
        getattr(parent, field)[idx] = new_node
    else:  # Node is a direct attribute
        setattr(parent, field, new_node)
    
    return original_ast

def generate_unified_diff(original_code, fixed_code, file_path):
    """Generate a unified diff between original and fixed code."""
    original_lines = original_code.splitlines(True)
    fixed_lines = fixed_code.splitlines(True)
    
    diff = difflib.unified_diff(
        original_lines,
        fixed_lines,
        fromfile=f'a/{file_path}',
        tofile=f'b/{file_path}',
        n=3
    )
    
    return ''.join(diff)

In [ ]:
def process_task(task, llm_name):
    """Processes a single task using the specified LLM with AST-based approach."""
    response = None
    try:
        instance_id = task["instance_id"]
        problem_statement = task["problem_statement"]
        patch = task["patch"]
        
        # Extract file path from patch
        file_path = extract_modified_file_path(patch)
        if not file_path:
            print(f"[Warning] No file path found for {instance_id}")
            return
        
        # Read file content
        file_full_path = f"./codebases/{instance_id}/{file_path}"
        if not os.path.exists(file_full_path):
            print(f"[Error] File not found: {file_full_path}")
            return
        
        with open(file_full_path, "r", encoding="utf-8") as f:
            file_content = f.read()
        
        # Parse file content into AST
        try:
            original_ast = ast.parse(file_content)
            ast_content = ast.dump(original_ast, include_attributes=True)
        except SyntaxError as e:
            print(f"[Error] Failed to parse {file_path}: {e}")
            return
        
        # Format prompt with AST content
        prompt = ast_prompt_template.format(
            problem_statement=problem_statement,
            ast_content=ast_content,
            file_path=file_path
        )
        
        # Get response from LLM with structured output
        llm = llms[llm_name]
        if isinstance(llm, BaseChatModel):
            response = llm.invoke([HumanMessage(content=prompt)]).content
        elif isinstance(llm, BaseLLM):
            response = llm.predict(prompt)
        else:
            raise ValueError(f"Unknown LLM type for {llm_name}")
        
        if '```json' in response:
            response = response.split('```json')[1].split('```')[-2]
        # response = response.lstrip('```json').lstrip('```').rstrip('```')


        # Parse the JSON response
        try:
            fixed_ast_strings = json.loads(response)
            if not isinstance(fixed_ast_strings, list):
                raise ValueError("Expected JSON array response")
        except json.JSONDecodeError as e:
            print(f"[Error] Failed to parse JSON from LLM response: {e}")
            # print(f"Raw response: {response}")
            # return response
            return
        
        # Apply each fixed AST to the original AST
        modified_ast = original_ast
        for fixed_ast_str in fixed_ast_strings:
            # Parse the fixed AST string
            try:
                fixed_ast = ast.parse(fixed_ast_str)
            except SyntaxError as e:
                print(f"[Error] Failed to parse fixed AST: {e}")
                continue
            
            # Find matching nodes in original AST
            matches = find_matching_node(modified_ast, fixed_ast_str)
            if not matches:
                print(f"[Warning] No matching node found for: {fixed_ast_str[:100]}...")
                continue
                
            # Replace the node in the AST
            for node, parent_info in matches:
                modified_ast = replace_node_in_ast(modified_ast, parent_info, fixed_ast)
        
        # Convert modified AST back to code
        fixed_code = astunparse.unparse(modified_ast)
        # print()
        # print(fixed_code)
        # print()
        
        # Generate unified diff
        diff_output = generate_unified_diff(file_content, fixed_code, file_path)
        
        # Save the diff
        output_path = f"./test_outputs/{llm_name}/ast_based/{instance_id}.diff"
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(diff_output)
        
        print(f"[Success] AST-based diff saved: {output_path}")
    except Exception as e:
        print(f"[Error] Failed to process {instance_id} with {llm_name}: {e}")
    
    # return response

In [ ]:
# Iterate through dataset and process each task
llm_name = 'chatgpt'
for index, task in enumerate(dataset):
    response = process_task(task, llm_name)